# Test: `ClassificationPicker`

This notebook uses two well-known datasets:

| Part | Dataset | Why it's a good test |
|---|---|---|
| **A** | **IBM Telco Customer Churn** (~7,000 customers) | Binary target (`Churn` Yes/No). Classes are uneven (about 27% churn), there are many text columns, and `TotalCharges` has hidden blanks to clean |
| **B** | **Palmer Penguins** (~340 penguins) | Three classes (species), missing values, and text columns (`island`, `sex`). Ranked by AUC |

Each part ends with automatic checks that print **All checks passed**.

In [1]:
# --- Find the ML root folder and import the pickers package ---
import sys
from pathlib import Path

ml_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pickers").is_dir())
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pickers import ClassificationPicker
print("Using pickers from:", ml_root / "pickers")

Using pickers from: C:\data\GitHub\Python\pickers


# Part A: Telco Customer Churn (binary)
## A1. Load the data

In [2]:
CHURN_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
churn_raw = pd.read_csv(CHURN_URL)
print(churn_raw.shape)
churn_raw.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## A2. Clean it
- `customerID` is just an identifier, so we drop it.
- `TotalCharges` *looks* numeric but was read as text, because 11 brand-new customers have a blank `" "` instead of a number.
  `isna()` won't find those, which makes it a good reminder to check dtypes, not just missing values.

In [3]:
print("TotalCharges dtype:", churn_raw["TotalCharges"].dtype)

churn = churn_raw.drop(columns="customerID")
churn["TotalCharges"] = pd.to_numeric(churn["TotalCharges"], errors="coerce")   # blanks -> NaN
print("Missing after conversion:", churn["TotalCharges"].isna().sum())

churn_unclean = churn.copy()            # kept for the 'rejects missing values' check
churn = churn.dropna().reset_index(drop=True)
print("Rows after cleaning:", len(churn))
churn["Churn"].value_counts(normalize=True).round(3)

TotalCharges dtype: str
Missing after conversion: 11
Rows after cleaning: 7032


Churn
No     0.734
Yes    0.266
Name: proportion, dtype: float64

## A3. Set up the picker
- `positive_class="Yes"`: Recall, Precision and F1 are measured on catching churners.
- `sort="F1"`: with uneven classes, accuracy is misleading. A model that predicts "No" for everyone is already about 73% accurate.
- The classes are uneven, so `class_weight="balanced"` should be switched on automatically as the starting value.

In [4]:
cp = ClassificationPicker(churn, target="Churn", positive_class="Yes", sort="F1")

print(cp.summary())
print()
for m in ["lr", "knn", "dt", "rf", "hgb"]:
    print(f"{m:5}", cp.starting_params(m))

Rows (train): 5,625   Features (encoded): 45   Classes: ['No', 'Yes']
Class balance (smallest/largest): 0.36   class_weight start: balanced
Categorical columns one-hot encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Positive class (for Recall/Precision/F1): 'Yes'

lr    {'class_weight': 'balanced', 'max_iter': 3000}
knn   {'n_neighbors': 25, 'weights': 'distance'}
dt    {'class_weight': 'balanced', 'min_samples_leaf': 50}
rf    {'class_weight': 'balanced', 'n_estimators': 300}
hgb   {'class_weight': 'balanced', 'early_stopping': True, 'learning_rate': 0.05, 'max_iter': 1000, 'n_iter_no_change': 20}


## A4. Compare all models (stratified 5-fold CV)

In [5]:
%%time
churn_lb = cp.compare_models()
churn_lb

2 classes, smallest/largest = 0.36 -> class_weight=balanced as starting value
  Dummy (majority class)           F1=0.0000  (1.4s)
  Logistic Regression              F1=0.6322  (1.1s)
  Ridge Classifier                 F1=0.6284  (0.1s)
  Linear SVM                       F1=0.6271  (0.1s)
  SGD Classifier                   F1=0.5818  (0.1s)
  Naive Bayes (Gaussian)           F1=0.6001  (0.1s)
  Linear Discriminant Analysis     F1=0.5890  (0.1s)
  Quadratic Discriminant Analysis  F1=0.6261  (0.1s)
  K Neighbors                      F1=0.5777  (0.2s)
  SVM (RBF kernel)                 F1=0.6231  (1.1s)
  Decision Tree                    F1=0.6231  (0.1s)
  Random Forest                    F1=0.6046  (1.4s)
  Extra Trees                      F1=0.5306  (1.7s)
  AdaBoost                         F1=0.5949  (1.5s)
  Gradient Boosting                F1=0.5839  (1.1s)
  Hist Gradient Boosting           F1=0.6387  (0.5s)
  XGBoost                          F1=0.6040  (0.5s)

Skipped: gpc (turbo 

,Model,Accuracy,AUC,Recall,Precision,F1,F1 SD,Kappa,MCC,Fit (s)
ID,,,,,,,,,,
hgb,Hist Gradient Boosting,0.7666,0.8452,0.7766,0.5426,0.6387,0.0073,0.4742,0.4910,0.3825
lr,Logistic Regression,0.7520,0.8459,0.8020,0.5218,0.6322,0.0070,0.4575,0.4817,0.0434
ridge,Ridge Classifier,0.7461,0.8433,0.8080,0.5143,0.6284,0.0091,0.4497,0.4764,0.0219
svm,Linear SVM,0.7461,0.8453,0.8033,0.5144,0.6271,0.0103,0.4484,0.4741,0.0415
qda,Quadratic Discriminant Analysis,0.7545,0.8351,0.7732,0.5261,0.6261,0.0125,0.4531,0.4717,0.0244
dt,Decision Tree,0.7433,0.8275,0.7980,0.5115,0.6231,0.0075,0.4425,0.4679,0.0326
rbfsvm,SVM (RBF kernel),0.7476,0.8261,0.7853,0.5165,0.6231,0.0081,0.4452,0.4674,0.6977
rf,Random Forest,0.7760,0.8256,0.6448,0.5693,0.6046,0.0189,0.4492,0.4510,1.1917
xgboost,XGBoost,0.7726,0.8301,0.6528,0.5624,0.6040,0.0093,0.4459,0.4485,0.4470


Look at the **Dummy** row: high accuracy, but an F1 of 0, because it never predicts a churner.
That's the accuracy trap with uneven classes.

## A5. Tune the top 3, then check on the holdout set

In [6]:
%%time
churn_tuned = cp.tune_top(n=3, n_iter=20)
churn_tuned

  Hist Gradient Boosting           start=0.6387  tuned=0.6423  -> keep tuned
  Logistic Regression              start=0.6322  tuned=0.6321  -> keep starting
  Ridge Classifier                 start=0.6284  tuned=0.6310  -> keep tuned
CPU times: total: 5.56 s
Wall time: 10.5 s


,Model,CV F1 (start),CV F1 (tuned),Kept,Holdout Accuracy,Holdout AUC,Holdout Recall,Holdout Precision,Holdout F1,Holdout Kappa,Holdout MCC,Tune (s)
ID,,,,,,,,,,,,
hgb,Hist Gradient Boosting,0.6387,0.6423,tuned,0.7370,0.8403,0.8048,0.5033,0.6193,0.4343,0.4623,8.5703
lr,Logistic Regression,0.6322,0.6321,starting,0.7257,0.8351,0.7968,0.4901,0.6069,0.4141,0.4429,0.7596
ridge,Ridge Classifier,0.6284,0.6310,tuned,0.7278,0.8320,0.7888,0.4925,0.6064,0.4149,0.4418,0.6057


In [7]:
print(cp.summary())

Rows (train): 5,625   Features (encoded): 45   Classes: ['No', 'Yes']
Class balance (smallest/largest): 0.36   class_weight start: balanced
Categorical columns one-hot encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Positive class (for Recall/Precision/F1): 'Yes'

Best model: Hist Gradient Boosting (hgb)
Holdout F1: 0.6193
Recommended hyperparameters:
  class_weight = balanced
  early_stopping = True
  learning_rate = 0.010491
  max_iter = 1000
  n_iter_no_change = 20
  l2_regularization = 0.002934
  max_features = 0.599421
  max_leaf_nodes = 15
  min_samples_leaf = 39


In [8]:
best_churn = cp.tuned_.index[0]
print(cp.class_report(best_churn))
cp.confusion_matrix(best_churn)

              precision    recall  f1-score   support

          No       0.91      0.71      0.80      1033
         Yes       0.50      0.80      0.62       374

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.80      0.74      0.75      1407



,pred No,pred Yes
actual No,736,297
actual Yes,73,301


## A6. Use the final model
`finalize()` refits on all rows, and its predictions come back as the original labels (`'Yes'` / `'No'`).

In [9]:
churn_model = cp.finalize()

sample = churn.drop(columns="Churn").head(8)
proba_yes = churn_model.predict_proba(sample)[:, list(churn_model.classes_).index("Yes")]
pd.DataFrame({"actual": churn["Churn"].head(8).values,
              "predicted": churn_model.predict(sample),
              "P(churn)": proba_yes.round(3)})

,actual,predicted,P(churn)
0,No,Yes,0.847
1,No,No,0.101
2,Yes,Yes,0.640
3,No,No,0.098
4,Yes,Yes,0.826
5,Yes,Yes,0.905
6,No,Yes,0.609
7,No,No,0.420


## A7. Automatic checks: churn

In [10]:
lb, tu = cp.leaderboard_, cp.tuned_
best = tu.index[0]

checks = {
    "leaderboard has 15+ models":                       len(lb) >= 15,
    "class_weight='balanced' switched on automatically": cp.ctx.balanced,
    "positive class is 'Yes'":                          cp.classes_[1] == "Yes",
    "accuracy trap: dummy accuracy > 0.70 but F1 = 0":  lb.loc["dummy", "Accuracy"] > 0.70 and lb.loc["dummy", "F1"] == 0,
    "best CV F1 beats dummy by 0.5+":                   lb["F1"].max() > lb.loc["dummy", "F1"] + 0.5,
    "15 text columns were one-hot encoded":             len(cp.cat_cols) == 15,
    "3 models were tuned, each with params":            len(tu) == 3 and all(m in cp.recommended_params_ for m in tu.index),
    "best holdout AUC is above 0.80":                   tu.loc[best, "Holdout AUC"] > 0.80,
    "confusion matrix covers the whole holdout set":    cp.confusion_matrix(best).values.sum() == len(cp.y_test),
    "final model predicts 'Yes'/'No' labels":           set(churn_model.predict(sample)) <= {"Yes", "No"},
    "missing values are rejected":                      False,
}
try:
    ClassificationPicker(churn_unclean, target="Churn")
except ValueError:
    checks["missing values are rejected"] = True

for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some churn checks failed - see above"
print("\nAll churn checks passed")

PASS  leaderboard has 15+ models
PASS  class_weight='balanced' switched on automatically
PASS  positive class is 'Yes'
PASS  accuracy trap: dummy accuracy > 0.70 but F1 = 0
PASS  best CV F1 beats dummy by 0.5+
PASS  15 text columns were one-hot encoded
PASS  3 models were tuned, each with params
PASS  best holdout AUC is above 0.80
PASS  confusion matrix covers the whole holdout set
PASS  final model predicts 'Yes'/'No' labels
PASS  missing values are rejected

All churn checks passed


# Part B: Palmer Penguins (three classes, ranked by AUC)
## B1. Load and clean

In [11]:
PENGUIN_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
penguins_raw = pd.read_csv(PENGUIN_URL)
print("Missing values:\n", penguins_raw.isna().sum()[penguins_raw.isna().sum() > 0])

penguins = penguins_raw.dropna().reset_index(drop=True)
print("\nRows after cleaning:", len(penguins))
penguins["species"].value_counts()

Missing values:
 bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Rows after cleaning: 333


species
Adelie       146
Gentoo       119
Chinstrap     68
Name: count, dtype: int64

## B2. Compare and tune
With `sort="AUC"`, models that can't give probability scores (Ridge Classifier, Linear SVM) are skipped with a note.
With three classes, AUC is averaged one-vs-rest and weighted by class size.

In [12]:
%%time
pp = ClassificationPicker(penguins, target="species", sort="AUC")
peng_lb = pp.compare_models()
peng_lb

3 classes, smallest/largest = 0.46 -> class_weight=balanced as starting value
  Dummy (majority class)           AUC=0.5000  (0.0s)
  Logistic Regression              AUC=1.0000  (0.0s)
  SGD Classifier                   AUC=1.0000  (0.0s)
  Naive Bayes (Gaussian)           AUC=0.9989  (0.0s)
  Linear Discriminant Analysis     AUC=1.0000  (0.0s)
  Quadratic Discriminant Analysis  AUC=1.0000  (0.0s)
  K Neighbors                      AUC=1.0000  (0.0s)
  SVM (RBF kernel)                 AUC=1.0000  (0.0s)
  Decision Tree                    AUC=0.9866  (0.0s)
  Random Forest                    AUC=0.9990  (0.3s)
  Extra Trees                      AUC=1.0000  (0.3s)
  AdaBoost                         AUC=1.0000  (0.3s)
  Gradient Boosting                AUC=0.9980  (0.3s)
  Hist Gradient Boosting           AUC=0.9999  (0.3s)
  XGBoost                          AUC=1.0000  (0.1s)

Skipped: ridge (no probability scores, so AUC can't be ranked), svm (no probability scores, so AUC can't be ran

,Model,Accuracy,AUC,AUC SD,Recall,Precision,F1,Kappa,MCC,Fit (s)
ID,,,,,,,,,,
lr,Logistic Regression,1.0000,1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0072
sgd,SGD Classifier,0.9887,1.0000,0.0000,0.9887,0.9895,0.9885,0.9822,0.9828,0.0125
lda,Linear Discriminant Analysis,0.9963,1.0000,0.0000,0.9963,0.9964,0.9962,0.9941,0.9943,0.0091
qda,Quadratic Discriminant Analysis,1.0000,1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.0067
knn,K Neighbors,0.9963,1.0000,0.0000,0.9963,0.9964,0.9962,0.9941,0.9943,0.0056
rbfsvm,SVM (RBF kernel),0.9963,1.0000,0.0000,0.9963,0.9964,0.9962,0.9941,0.9943,0.0072
et,Extra Trees,1.0000,1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,0.1975
ada,AdaBoost,0.9925,1.0000,0.0000,0.9925,0.9928,0.9924,0.9882,0.9884,0.1997
xgboost,XGBoost,0.9887,1.0000,0.0000,0.9887,0.9894,0.9887,0.9823,0.9826,0.0708


In [13]:
peng_tuned = pp.tune_top(n=5, n_iter=15)
print(pp.summary())
print()
print(pp.class_report(pp.tuned_.index[0]))
pp.confusion_matrix(pp.tuned_.index[0])

  Logistic Regression              start=1.0000  tuned=1.0000  -> keep starting
  SGD Classifier                   start=1.0000  tuned=1.0000  -> keep starting
  Linear Discriminant Analysis     start=1.0000  tuned=1.0000  -> keep starting
  Quadratic Discriminant Analysis  start=1.0000  tuned=1.0000  -> keep starting
  K Neighbors                      start=1.0000  tuned=1.0000  -> keep starting
Rows (train): 266   Features (encoded): 9   Classes: ['Adelie', 'Chinstrap', 'Gentoo']
Class balance (smallest/largest): 0.46   class_weight start: balanced
Categorical columns one-hot encoded: ['island', 'sex']

Best model: Logistic Regression (lr)
Holdout AUC: 1.0000
Recommended hyperparameters:
  class_weight = balanced
  max_iter = 3000

              precision    recall  f1-score   support

      Adelie       1.00      0.97      0.98        29
   Chinstrap       0.93      1.00      0.97        14
      Gentoo       1.00      1.00      1.00        24

    accuracy                          

,pred Adelie,pred Chinstrap,pred Gentoo
actual Adelie,28,1,0
actual Chinstrap,0,14,0
actual Gentoo,0,0,24


## B3. Automatic checks: penguins

In [14]:
lb, tu = pp.leaderboard_, pp.tuned_
best = tu.index[0]
species = {"Adelie", "Chinstrap", "Gentoo"}
peng_model = pp.finalize()

checks = {
    "3 classes found":                              pp.ctx.k == 3 and set(pp.classes_) == species,
    "AUC sort skipped Ridge and Linear SVM":        {"ridge", "svm"} <= set(pp.skipped_),
    "island and sex were one-hot encoded":          set(pp.cat_cols) == {"island", "sex"},
    "best holdout accuracy is above 0.95":          tu.loc[best, "Holdout Accuracy"] > 0.95,
    "confusion matrix is 3 x 3":                    pp.confusion_matrix(best).shape == (3, 3),
    "final model predicts species names":           set(peng_model.predict(penguins.drop(columns="species"))) <= species,
    "probabilities sum to 1":                       np.allclose(peng_model.predict_proba(penguins.drop(columns="species")).sum(axis=1), 1),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
assert all(checks.values()), "Some penguin checks failed - see above"
print("\nAll checks passed")

PASS  3 classes found
PASS  AUC sort skipped Ridge and Linear SVM
PASS  island and sex were one-hot encoded
PASS  best holdout accuracy is above 0.95
PASS  confusion matrix is 3 x 3
PASS  final model predicts species names
PASS  probabilities sum to 1

All checks passed
